In [1]:
# --- Environment Setup (do not edit) ---
import os, sys
from pathlib import Path


def _detect_platform():
    try:
        import google.colab  # noqa: F401

        return "colab", False
    except ImportError:
        pass
    if Path("/workspace").exists() and os.environ.get("VAST_CONTAINERLABEL"):
        return "vastai", False
    if Path("/workspace").exists():
        return "vastai", True
    if sys.platform == "win32":
        return "windows", False
    if sys.platform == "darwin":
        return "mac", False
    return None, True


PLATFORM, _uncertain = _detect_platform()

if PLATFORM == "colab":
    from google.colab import drive

    drive.mount("/content/drive")

try:
    _nb_path = Path(__file__).resolve()
except NameError:
    _nb_path = Path.cwd()

if PLATFORM == "colab":
    PROJECT_ROOT = Path("/content/drive/MyDrive/Thesis_Final/fake-news-detection")
else:
    PROJECT_ROOT = _nb_path.parents[1]

sys.path.insert(0, str(PROJECT_ROOT))

_env_map = {
    "colab": PROJECT_ROOT / ".env.colab",
    "vastai": PROJECT_ROOT / ".env.vastai",
    "windows": PROJECT_ROOT / ".env.windows",
    "mac": PROJECT_ROOT / ".env.mac",
}

if PLATFORM is None:
    print(
        "\u26a0\ufe0f  WARNING: Could not detect platform. Falling back to .env (local)."
    )
    _env_file = PROJECT_ROOT / ".env"
elif _uncertain:
    print(
        f"\u26a0\ufe0f  WARNING: Detected /workspace but VAST_CONTAINERLABEL is not set. Assuming Vast.ai."
    )
    _env_file = _env_map["vastai"]
else:
    _env_file = _env_map[PLATFORM]

from dotenv import load_dotenv

if not _env_file.exists():
    _fallback = PROJECT_ROOT / ".env"
    print(f"\u26a0\ufe0f  WARNING: Expected env file not found: {_env_file}")
    if _fallback.exists():
        print(f"   Falling back to: {_fallback}")
        _env_file = _fallback
    else:
        raise FileNotFoundError(
            f"No .env file found. Copy the correct example file:\n"
            f"  cp .env.{PLATFORM or 'mac'}.example .env.{PLATFORM or 'mac'}"
        )

load_dotenv(_env_file, override=True)

from src.utils.env_utils import get_data_root

DATA_ROOT = get_data_root()

print(f"\u2705 Platform : {PLATFORM or 'unknown (local fallback)'}")
print(f"\u2705 Env file : {_env_file}")
print(f"\u2705 DATA_ROOT: {DATA_ROOT}")
print(
    f"{'\u2705' if DATA_ROOT.exists() else '\u274c'} Path exists: {DATA_ROOT.exists()}"
)
# ---------------------------------------------------------------------------

✅ Platform : mac
✅ Env file : /Users/haila/My File/projects/fake-new-detection/.env.mac
✅ DATA_ROOT: /Users/haila/Library/CloudStorage/GoogleDrive-latruonghai@gmail.com/My Drive/Thesis_Final/fake-news-data-for-thesis
✅ Path exists: True


# ViFactCheck Original PLM Training (Stage 3.9 — Original)

Fine-tunes **`vinai/phobert-large`** following the **original ViFactCheck** paper (AAAI 2025) training code.

This notebook faithfully reproduces the original training setup from
[`TTHHA/ViFactCheck`](https://github.com/TTHHA/ViFactCheck) `src/classification/main_results/plm_training.py`:

-   **Model**: `vinai/phobert-large` (1024-dim hidden)
-   **Architecture**: Pooler output → Dropout(0.3) → Linear (NOT CLS token)
-   **Input**: `Statement + Context` sentence pair (matching original code)
-   **Loss**: Plain `CrossEntropyLoss()` (no class weights, no label smoothing)
-   **Optimizer**: `AdamW(lr=5e-6)` — no scheduler, no warmup, no weight decay, no grad clip
-   **Training**: Fixed 10 epochs, no early stopping
-   **Classification**: 3-class (Supported=0, Refuted=1, NEI=2)
-   **Tokenization**: `truncation=True` (longest_first), `max_length=256`, `batch_size=16`

**Data**: HuggingFace `tranthaihoa/vifactcheck` (train/dev/test splits as provided)

```
Output: training/checkpoints_vifactcheck_original/<run>/best_model.pth + tokenizer/
```


In [2]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent if Path.cwd().name == "pipeline" else Path.cwd()

try:
    from dotenv import load_dotenv

    load_dotenv(PROJECT_ROOT / ".env", override=False)
except ImportError:
    pass

DATA_ROOT = (
    Path(os.environ["DATA_ROOT"]) if os.environ.get("DATA_ROOT") else PROJECT_ROOT
)

CONFIG = {
    "paths": {
        "checkpoint_root": DATA_ROOT / "training" / "checkpoints_vifactcheck_original",
        "mlflow_dir": DATA_ROOT / "mlruns",
    },
    "data": {
        "hf_dataset": "tranthaihoa/vifactcheck",
        # Original code uses Statement + Context as sentence pair
        "text_field": "Statement",
        "context_field": "Context",  # 'Context' = full article (original plm_training.py)
        "label_field": "labels",
        "max_length": 256,
        "num_classes": 3,  # 3-class: Supported(0), Refuted(1), NEI(2)
    },
    "model": {
        "backbone": "vinai/phobert-large",  # original uses phobert-large, NOT phobert-base-v2
        "dropout": 0.3,
        "num_classes": 3,
    },
    "training": {
        "batch_size": 16,
        "epochs": 10,  # fixed 10 epochs, no early stopping (original)
        "lr": 0.5e-5,  # = 5e-6 (original)
        "seed": 28,  # original uses random_state=28 for data split
    },
    "safety": {
        "smoke_test": False,
        "smoke_batches": 2,
    },
}

print(f"Project root: {PROJECT_ROOT}")
print(f"Data root:    {DATA_ROOT}")
print(f"Checkpoint:   {CONFIG['paths']['checkpoint_root']}")
print(f"Backbone:     {CONFIG['model']['backbone']}")

Project root: /Users/haila/My File/projects/fake-new-detection
Data root:    /Users/haila/Library/CloudStorage/GoogleDrive-latruonghai@gmail.com/My Drive/Thesis_Final/fake-news-data-for-thesis
Checkpoint:   /Users/haila/Library/CloudStorage/GoogleDrive-latruonghai@gmail.com/My Drive/Thesis_Final/fake-news-data-for-thesis/training/checkpoints_vifactcheck_original
Backbone:     vinai/phobert-large


In [3]:
# --- DEPENDENCY PREFLIGHT ---
import importlib, sys

_required = {
    "torch": "torch",
    "transformers": "transformers",
    "datasets": "datasets",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "tqdm": "tqdm",
    "sklearn": "scikit-learn",
}

_missing = []
for mod, pkg in _required.items():
    try:
        importlib.import_module(mod)
    except ImportError:
        _missing.append(pkg)

if _missing:
    print(f"Missing packages: {_missing}")
    print("Install with:  pip install transformers datasets scikit-learn seaborn tqdm")
    raise RuntimeError(f"Missing packages: {_missing}.")
else:
    print("All dependencies satisfied.")

All dependencies satisfied.


In [4]:
# --- IMPORTS AND SETUP ---
import os, gc, json, random
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from transformers import AutoTokenizer, AutoModel

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG["paths"]["checkpoint_root"].mkdir(parents=True, exist_ok=True)

print(f"PyTorch      : {torch.__version__}")
import transformers

print(f"Transformers : {transformers.__version__}")


def select_device():
    if torch.cuda.is_available():
        dev = torch.device("cuda")
        mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"Device: cuda ({torch.cuda.get_device_name(0)}, {mem:.1f} GB)")
    elif torch.backends.mps.is_available():
        dev = torch.device("mps")
        print("Device: mps (Apple Silicon). For full training, prefer a CUDA GPU.")
    else:
        dev = torch.device("cpu")
        print("Device: cpu. Use smoke_test=True for local validation.")
    return dev


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    print(f"Seed: {seed}")


DEVICE = select_device()
seed_everything(CONFIG["training"]["seed"])

PyTorch      : 2.12.0
Transformers : 5.10.2
Device: mps (Apple Silicon). For full training, prefer a CUDA GPU.
Seed: 28


## Step 1: Load ViFactCheck Dataset from HuggingFace

Loads `tranthaihoa/vifactcheck` with `Statement` + `Context` as sentence pairs.
Labels: 0=Supported, 1=Refuted, 2=NEI (3-class, matching original paper).


In [5]:
from datasets import load_dataset

hf_name = CONFIG["data"]["hf_dataset"]
text_field = CONFIG["data"]["text_field"]
ctx_field = CONFIG["data"]["context_field"]
label_field = CONFIG["data"]["label_field"]

print(f"Loading '{hf_name}' from HuggingFace...")
raw = load_dataset(hf_name)

available = list(raw.keys())
print(f"  Available splits: {available}")
sample = dict(raw[available[0]][0])
print(f"  Columns: {list(sample.keys())}")
print(f"  First sample: { {k: str(v)[:60] for k, v in sample.items()} }")

print(f"\n  text_field    = {text_field!r}  (claim)")
print(f"  context_field = {ctx_field!r}  (full article context)")
print(f"  label_field   = {label_field!r}")

# Build examples: (Statement, Context, label) — matching original plm_training.py
raw_examples = {}
for hf_split in available:
    ds = raw[hf_split]
    examples = []
    for item in ds:
        stmt = str(item.get(text_field) or "").strip()
        ctx = str(item.get(ctx_field) or "").strip()
        label = int(item.get(label_field))
        if not stmt:
            continue
        examples.append({"text": stmt, "text_b": ctx, "label": label})
    nc = CONFIG["data"]["num_classes"]
    hist = [sum(1 for e in examples if e["label"] == i) for i in range(nc)]
    print(f"  {hf_split:5s}: {len(examples):5d} examples | label dist {hist}")
    raw_examples[hf_split] = examples

print(f"\nData loaded: { {k: len(v) for k, v in raw_examples.items()} }")

Loading 'tranthaihoa/vifactcheck' from HuggingFace...


  Available splits: ['train', 'test', 'dev']
  Columns: ['Unnamed: 0', 'index', 'Statement', 'Context', 'annotation_id', 'Topic', 'Author', 'Url', 'labels', 'Evidence']
  First sample: {'Unnamed: 0': '0', 'index': '3049', 'Statement': 'Phó Thủ tướng Trần Hồng Hà thay mặt Chính phủ, Thủ tướng Chí', 'Context': '(Chinhphu.vn) - Đây là mong muốn, gửi gắm của Phó Thủ tướng ', 'annotation_id': '18933775', 'Topic': 'Chính trị', 'Author': 'Chính Phủ', 'Url': 'https://baochinhphu.vn/lan-toa-nhung-gia-tri-van-hoa-tot-dep', 'labels': '0', 'Evidence': 'Thay mặt Chính phủ, Thủ tướng Chính phủ, Phó Thủ tướng Trần '}

  text_field    = 'Statement'  (claim)
  context_field = 'Context'  (full article context)
  label_field   = 'labels'
  train:  5062 examples | label dist [1751, 1658, 1653]
  test :  1447 examples | label dist [508, 468, 471]
  dev  :   723 examples | label dist [256, 244, 223]

Data loaded: {'train': 5062, 'test': 1447, 'dev': 723}


## Step 2: Tokenize and Build DataLoaders

Uses `tokenizer()` with `truncation=True` (longest_first) matching the original code.
`return_token_type_ids=False`, `padding="max_length"`.


In [6]:
class SentencePairDataset(Dataset):
    """Matches original ViFactCheck SentencePairDataset."""

    def __init__(self, sentence_pairs, labels, tokenizer, max_length):
        self.sentence_pairs = sentence_pairs
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.sentence_pairs)

    def __getitem__(self, idx):
        sentence1, sentence2 = self.sentence_pairs[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(
            sentence1,
            text_pair=sentence2,
            add_special_tokens=True,
            max_length=self.max_length,
            return_token_type_ids=False,
            padding="max_length",
            return_attention_mask=True,
            return_tensors="pt",
            truncation=True,  # longest_first (original)
        )
        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "label": torch.tensor(label, dtype=torch.long),
        }


backbone = CONFIG["model"]["backbone"]
print(f"Loading tokenizer: {backbone}")
tokenizer = AutoTokenizer.from_pretrained(backbone)
print("Tokenizer loaded.")

_max_len = CONFIG["data"]["max_length"]
_bs = CONFIG["training"]["batch_size"]
_smoke = CONFIG["safety"]["smoke_test"]
_smoke_n = CONFIG["safety"]["smoke_batches"]

# Build sentence pair lists: [(stmt, ctx), ...] and [label, ...]
datasets_map = {}
for split, exs in raw_examples.items():
    pairs = [(e["text"], e["text_b"]) for e in exs]
    labels = [e["label"] for e in exs]
    datasets_map[split] = SentencePairDataset(pairs, labels, tokenizer, _max_len)


def _make_loader(ds, shuffle, batch_size):
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0)


loaders = {
    "train": _make_loader(datasets_map["train"], shuffle=True, batch_size=_bs),
    "dev": _make_loader(
        datasets_map.get("dev", datasets_map["train"]), shuffle=False, batch_size=_bs
    ),
    "test": _make_loader(
        datasets_map.get("test", datasets_map["train"]), shuffle=False, batch_size=_bs
    ),
}

# Smoke-test wrapper
if _smoke:
    from itertools import islice

    class _SmokeLoader:
        def __init__(self, loader, n):
            self._l = loader
            self._n = n

        def __iter__(self):
            return islice(self._l, self._n)

        def __len__(self):
            return min(self._n, len(self._l))

    loaders = {k: _SmokeLoader(v, _smoke_n) for k, v in loaders.items()}
    print(f"SMOKE TEST: {_smoke_n} batches per split")

# Sanity check batch
_b = next(iter(loaders["train"]))
print(
    f"\nBatch shapes -- input_ids: {tuple(_b['input_ids'].shape)}  "
    f"attention_mask: {tuple(_b['attention_mask'].shape)}  "
    f"label: {tuple(_b['label'].shape)}"
)
for split in ["train", "dev", "test"]:
    _ds = datasets_map.get(split)
    if _ds:
        nc = CONFIG["data"]["num_classes"]
        hist = [sum(1 for l in _ds.labels if l == i) for i in range(nc)]
        print(f"  {split:5s}: {len(_ds):5d} samples | label dist {hist}")

Loading tokenizer: vinai/phobert-large
Tokenizer loaded.


[transformers] Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
[transformers] Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
[transformers] Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
[transformers] Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
[transformers] Be aware, overflowing


Batch shapes -- input_ids: (16, 256)  attention_mask: (16, 256)  label: (16,)
  train:  5062 samples | label dist [1751, 1658, 1653]
  dev  :   723 samples | label dist [256, 244, 223]
  test :  1447 samples | label dist [508, 468, 471]


## Step 3: Build PhoBERT Classifier (Original Architecture)

Matches original `PhoBERTClassifier` from `plm_training.py`:

-   Uses **pooler output** (`pooled_output`), NOT CLS token
-   `Dropout(0.3)` → `Linear(hidden_size, num_classes)`
-   `phobert-large` has hidden_size=1024


In [7]:
class PhoBERTClassifier(nn.Module):
    """Original ViFactCheck PhoBERTClassifier — uses pooler output, not CLS token."""

    def __init__(self, phobert, num_classes):
        super(PhoBERTClassifier, self).__init__()
        self.phobert = phobert
        self.dropout = nn.Dropout(0.3)
        self.linear = nn.Linear(self.phobert.config.hidden_size, num_classes)
        self.hidden_size = self.phobert.config.hidden_size

    def forward(self, input_ids, attention_mask):
        _, pooled_output = self.phobert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=False,
        )
        dropout_output = self.dropout(pooled_output)
        logits = self.linear(dropout_output)
        return logits


print(f"Building model: {backbone}")
phobert = AutoModel.from_pretrained(backbone)
model = PhoBERTClassifier(phobert, num_classes=CONFIG["model"]["num_classes"]).to(
    DEVICE
)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters -- total: {total:,}  trainable: {trainable:,}")
print(
    f"Hidden size: {model.hidden_size}  ->  linear -> {CONFIG['model']['num_classes']} classes"
)

# Sanity forward
model.eval()
with torch.no_grad():
    _ids = _b["input_ids"][:2].to(DEVICE)
    _mask = _b["attention_mask"][:2].to(DEVICE)
    _logit = model(_ids, _mask)
print(
    f"Sanity forward output shape: {tuple(_logit.shape)}  (expected [2, {CONFIG['model']['num_classes']}])"
)

Building model: vinai/phobert-large


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Parameters -- total: 369,166,339  trainable: 369,166,339
Hidden size: 1024  ->  linear -> 3 classes
Sanity forward output shape: (2, 3)  (expected [2, 3])


## Step 4: Loss and Optimizer (Original)

Plain `CrossEntropyLoss()` + `AdamW(lr=5e-6)`. No scheduler, no warmup, no weight decay, no grad clip.
Matches original `plm_training.py` exactly.


In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["training"]["lr"])

print(f"Loss      : CrossEntropyLoss (plain, no class weights)")
print(f"Optimizer : AdamW  lr={CONFIG['training']['lr']}")
print(f"Epochs    : {CONFIG['training']['epochs']}  (fixed, no early stopping)")

Loss      : CrossEntropyLoss (plain, no class weights)
Optimizer : AdamW  lr=5e-06
Epochs    : 10  (fixed, no early stopping)


## Step 5: Run Directory Setup


In [9]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = f"vifactcheck_original_{timestamp}"

run_dir = CONFIG["paths"]["checkpoint_root"] / run_name
run_dir.mkdir(parents=True, exist_ok=True)
print(f"Run dir: {run_dir}")

Run dir: /Users/haila/Library/CloudStorage/GoogleDrive-latruonghai@gmail.com/My Drive/Thesis_Final/fake-news-data-for-thesis/training/checkpoints_vifactcheck_original/vifactcheck_original_20260718_002715


## Step 6: Training Loop (Original)

Fixed 10 epochs, no early stopping. Evaluates on dev set after each epoch.
Saves best checkpoint by dev macro-F1 (for convenience — original code doesn't save checkpoints,
but we need one for downstream use).


In [ ]:
def compute_metrics(y_true, y_pred, nc):
    acc = float(accuracy_score(y_true, y_pred))
    macro_f1 = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    per_f1 = f1_score(y_true, y_pred, average=None, zero_division=0)
    return {
        "accuracy": round(acc, 4),
        "macro_f1": round(macro_f1, 4),
        "per_f1": [round(float(f), 4) for f in per_f1],
    }


best_val_f1 = -1.0
best_epoch = -1
history = []
best_ckpt_path = run_dir / "best_model.pth"

epochs = CONFIG["training"]["epochs"]

for epoch in range(epochs):
    # --- Train ---
    model.train()
    train_loss = 0.0
    n_batches = 0
    pbar = tqdm(loaders["train"], desc=f"Epoch {epoch+1}/{epochs} [train]", leave=False)
    for batch in pbar:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        n_batches += 1
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    mean_train_loss = train_loss / max(1, n_batches)

    # --- Eval on dev ---
    model.eval()
    predictions, true_labels = [], []
    pbar = tqdm(loaders["dev"], desc=f"Epoch {epoch+1}/{epochs} [dev]", leave=False)
    for batch in pbar:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        with torch.no_grad():
            outputs = model(input_ids, attention_mask)
            _, predicted = torch.max(outputs, 1)
            predictions.extend(predicted.cpu().numpy().tolist())
            true_labels.extend(labels.cpu().numpy().tolist())

    dev_metrics = compute_metrics(
        true_labels, predictions, CONFIG["data"]["num_classes"]
    )
    print(
        f"Epoch {epoch+1}/{epochs}  train_loss={mean_train_loss:.4f}  "
        f"dev_acc={dev_metrics['accuracy']:.4f}  dev_macro_f1={dev_metrics['macro_f1']:.4f}"
    )

    history.append(
        {
            "epoch": epoch + 1,
            "train_loss": round(mean_train_loss, 4),
            "dev_accuracy": dev_metrics["accuracy"],
            "dev_macro_f1": dev_metrics["macro_f1"],
        }
    )

    # Save best checkpoint (by dev macro_f1)
    if dev_metrics["macro_f1"] > best_val_f1:
        best_val_f1 = dev_metrics["macro_f1"]
        best_epoch = epoch + 1
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "epoch": best_epoch,
                "dev_metrics": dev_metrics,
                "backbone": CONFIG["model"]["backbone"],
                "hidden_size": model.hidden_size,
                "num_classes": CONFIG["model"]["num_classes"],
            },
            best_ckpt_path,
        )
        print(
            f"  -> New best (dev_macro_f1={best_val_f1:.4f}), saved to {best_ckpt_path}"
        )

print(f"\nTraining complete. Best epoch: {best_epoch}  dev_macro_f1: {best_val_f1:.4f}")

Epoch 1/10 [train]:   0%|          | 0/317 [00:00<?, ?it/s]

[transformers] Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
[transformers] Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
[transformers] Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
[transformers] Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
[transformers] Be aware, overflowing

## Step 7: Final Test Evaluation

Loads the best checkpoint and runs on the test set.


In [ ]:
if not best_ckpt_path.exists():
    print("best_model.pth not found -- skipping test evaluation.")
else:
    _ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
    model.load_state_dict(_ckpt["model_state_dict"])
    print(f"Loaded best checkpoint (epoch {_ckpt['epoch']}) for test evaluation.")

    model.eval()
    predictions, true_labels = [], []
    pbar = tqdm(loaders["test"], desc="Test", leave=False)
    for batch in pbar:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        with torch.no_grad():
            outputs = model(input_ids, attention_mask)
            _, predicted = torch.max(outputs, 1)
            predictions.extend(predicted.cpu().numpy().tolist())
            true_labels.extend(labels.cpu().numpy().tolist())

    test_metrics = compute_metrics(
        true_labels, predictions, CONFIG["data"]["num_classes"]
    )
    print(f"\nTest Results:")
    print(f"  Accuracy  : {test_metrics['accuracy']:.4f}")
    print(f"  Macro F1  : {test_metrics['macro_f1']:.4f}")
    print(f"  Per-class F1: {test_metrics['per_f1']}")
    print()
    print(classification_report(true_labels, predictions, digits=4))

    # Save test report
    report_path = run_dir / "test_report.json"
    with open(report_path, "w") as f:
        json.dump(
            {
                "test_metrics": test_metrics,
                "best_epoch": best_epoch,
                "best_val_f1": best_val_f1,
            },
            f,
            indent=2,
        )
    print(f"Test report saved: {report_path}")

## Step 8: Visualize Training Curves and Confusion Matrix


In [ ]:
if not history:
    print("No training history to plot.")
else:
    sns.set_theme(style="whitegrid", palette="muted")
    hist_df = pd.DataFrame(history)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Loss curve
    axes[0].plot(hist_df["epoch"], hist_df["train_loss"], marker="o", color="steelblue")
    axes[0].set_title("Training Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")

    # Dev accuracy
    axes[1].plot(
        hist_df["epoch"], hist_df["dev_accuracy"], marker="s", color="seagreen"
    )
    axes[1].axvline(
        x=best_epoch,
        color="red",
        linestyle="--",
        alpha=0.5,
        label=f"Best (epoch {best_epoch})",
    )
    axes[1].set_title("Dev Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()

    # Dev macro F1
    axes[2].plot(hist_df["epoch"], hist_df["dev_macro_f1"], marker="^", color="coral")
    axes[2].axvline(
        x=best_epoch,
        color="red",
        linestyle="--",
        alpha=0.5,
        label=f"Best (epoch {best_epoch})",
    )
    axes[2].set_title("Dev Macro F1")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("Macro F1")
    axes[2].legend()

    plt.tight_layout()
    plt.savefig(run_dir / "training_curves.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Training curves saved: {run_dir / 'training_curves.png'}")

    # Confusion matrix (test set)
    if "true_labels" in dir() and "predictions" in dir():
        cm = confusion_matrix(true_labels, predictions)
        class_names = ["Supported", "Refuted", "NEI"]
        fig, ax = plt.subplots(figsize=(7, 6))
        sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=class_names,
            yticklabels=class_names,
            ax=ax,
        )
        ax.set_title("Test Confusion Matrix")
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        plt.tight_layout()
        plt.savefig(run_dir / "confusion_matrix.png", dpi=150, bbox_inches="tight")
        plt.show()
        print(f"Confusion matrix saved: {run_dir / 'confusion_matrix.png'}")

## Step 9: Save Tokenizer and Checkpoint Manifest


In [ ]:
# Save tokenizer alongside checkpoint
tokenizer_dir = run_dir / "tokenizer"
tokenizer.save_pretrained(str(tokenizer_dir))
print(f"Tokenizer saved: {tokenizer_dir}")

# Write manifest
manifest = {
    "best_checkpoint_path": str(best_ckpt_path),
    "tokenizer_path": str(tokenizer_dir),
    "best_epoch": best_epoch,
    "best_dev_macro_f1": best_val_f1,
    "backbone": CONFIG["model"]["backbone"],
    "hidden_size": model.hidden_size,
    "num_classes": CONFIG["model"]["num_classes"],
    "input_format": "Statement + Context (sentence pair)",
    "architecture": "pooler_output -> Dropout(0.3) -> Linear",
    "training_setup": {
        "loss": "CrossEntropyLoss (plain)",
        "optimizer": "AdamW",
        "lr": CONFIG["training"]["lr"],
        "epochs": CONFIG["training"]["epochs"],
        "batch_size": CONFIG["training"]["batch_size"],
        "max_length": CONFIG["data"]["max_length"],
        "scheduler": "none",
        "early_stopping": False,
        "class_weights": False,
    },
    "data_source": "HuggingFace tranthaihoa/vifactcheck",
    "label_scheme": "3-class: Supported(0), Refuted(1), NEI(2)",
    "note": "Follows original ViFactCheck plm_training.py (AAAI 2025)",
}
if "test_metrics" in dir():
    manifest["test_metrics"] = test_metrics

manifest_path = run_dir / "checkpoint_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)
print(f"Manifest written: {manifest_path}")
print(f"\nDone. All outputs in: {run_dir}")